## Task 1: Generate new sentences

In [2]:
from collections import defaultdict
from ast import literal_eval  # to convert string list ↦ python list
import random

def sample_next_token(prob_dict, prefix):
	# In the probability dictionary, get possible next tokens for the given prefix
	next_tokens = prob_dict.get(prefix, {})
	words = list(next_tokens.keys())
	probs = list(next_tokens.values())
	
	next_word = random.choices(words, weights=probs, k=1)[0]
	#print(f"From '{prefix}' sampled next word: '{next_word}'")
	return next_word

def generative_ngram(n, mt, prob_json, nr_outputs=1):
	# Read probability dictionary from the JSON file
	prob_dict = {}
	for prefix_str, next_tokens in prob_json.items():
		prefix = literal_eval(prefix_str)
		prob_dict[prefix] = {}
		for token, prob in next_tokens.items():
			prob_dict[prefix][token] = prob

	# Initialize output list
	output = [""] * nr_outputs

	for i in range(nr_outputs):
		if mt == "word":
			# Generate a sentence starting from <s> and ending at </s> based on the probability dictionary
			sentence = ["<s>"]
			
			# As long as the end symbol </s> is not reached, keep sampling the next word
			current_prefix = ()
			while sentence[-1] != "</s>":
				# Get the current prefix (last n-1 words)
				if n > 1:
					current_prefix = tuple(sentence[-(n-1):])
				next_word = sample_next_token(prob_dict, current_prefix)
				sentence.append(next_word)
			
			output[i] = " ".join(sentence)		# Convert list of generated words to a single string using space as separator

		else:
			# Generate a character sequence ending at ".", "!" or "?" based on the probability dictionary
			sequence = [""]
			# As long as none of the end symbols is reached, keep sampling the next character
			current_prefix = ()
			while sequence[-1] != "." and sequence[-1] != "!" and sequence[-1] != "?":
				# Get the current prefix (last n-1 characters)
				if n > 1:
					current_prefix = tuple(sequence[-(n-1):])
				next_char = sample_next_token(prob_dict, current_prefix)
				sequence.append(next_char)
			
			output[i] = "".join(sequence)		# Convert list of generated characters to a single string

	return output

In [3]:
import os
import json

#N = {1, 2, 3, 4, 5}
N = {2}					# 3+ grams currently not possible because of missing starting sequence (<s> --> ??? --> ??? --> ...)
							# The first words could be generated randomly or based on unigram probabilities?
							# How to deal with missing prefices in the probability dictionary?

#modeltype = {"word", "character"}
modeltype = {"word"}
#modeltype = {"character"}

category = "all"					# "business", "entertainment", "politics", "sport", "tech", "all"

sourcefolder = "./../Probability_NGrams/"
targetfolder = f"./../Task1_GeneratedSequences/"

nr_outputs = 1

for n in N:
	for mt in modeltype:
		# Read in JSON frequency file and catch non-existing files
		json_path = f"{sourcefolder}{mt}_{n}grams_probability.json"		# CATEGORY IGNORED --> currently "all"
		if not os.path.exists(json_path):
			print(f"Skipping (file not found): {json_path}")
			continue
		
		with open(json_path, "r", encoding="utf-8") as f:
			prob_json = json.load(f)

		# Let n-gram model create sequences
		output = generative_ngram(n, mt, prob_json, nr_outputs)

		# Save outputs as TXT file
		output_path = f"{targetfolder}{category}_{mt}_{n}grams_generated.txt"
		with open(output_path, "w", encoding="utf-8") as f:
			f.write(str(output))

### Task 2: Gap-Filler

Fill "< gap >" tokens in a sentence using an n-gram model.  

In [4]:
# Function to load probability dictionary from JSON, transforms JSON file to dictionary

def load_prob_Ndict(prob_json):
    prob_dict = {}
    for prefix_str, next_tokens in prob_json.items():
        prefix = literal_eval(prefix_str)
        prob_dict[prefix] = {}
        for token, prob in next_tokens.items():
            prob_dict[prefix][token] = prob
    return prob_dict

def load_prob_Unidict(prob_json):
    prob_dict = {}
    for prefix_str, next_tokens in prob_json.items():
        prefix = literal_eval(prefix_str)
        prob_dict[prefix] = next_tokens
    return prob_dict

In [5]:
import random 

def fill_gaps_in_sentence(sentence, n, nprob_dict, uniprob_dict):
    
    filled_sentence = sentence.copy()
    
    for i, token in enumerate(filled_sentence):
        if token != "<gap>":
            continue
        
        if n == 1: # if unigram, don't need prefix
            prefix = ()  
        else:
            # where the prefix should begin in the sentence, for an n-gram model, the prefix must contain n-1 previous words
            start = max(0, i-(n-1)) # index should not be negative
            prefix = tuple(filled_sentence[start:i])
        
        # look up probabilities for this prefix
        next_tokens = nprob_dict.get(prefix, {})
        
        # if no data available for this prefix, use unigram probabilities
        if not next_tokens:
            uni_tokens = uniprob_dict.get((), {})
            
            if uni_tokens:
                # sample randomly according to unigram probabilities
                words = list(uni_tokens.keys())
                probs = list(uni_tokens.values())
                best_word = random.choices(words, weights=probs, k=1)[0]
            else:
                best_word = "<unk>"
        
        else:
            # normal n-gram case: choose highest-prob word
            best_word = max(next_tokens, key=next_tokens.get)
        
        filled_sentence[i] = best_word # replaces the current <gap> with the best word (word with highest probability)
    
    return filled_sentence

- For each gap, the code first finds the prefix: if it’s a unigram model it uses an empty prefix, otherwise it takes the previous n−1 words as the prefix.

- It looks up which words can follow that prefix in the probability dictionary; if nothing is found, it falls back to the unigram probabilities.

- When the code falls back to unigram probabilities, it uses random sampling so that more frequent words are more likely—but not guaranteed—to be chosen, creating more natural and varied gap-fill results.

In [6]:
import json

with open("./../Probability_NGrams/word_1grams_probability.json", "r", encoding="utf-8") as f:
    prob_bigram_json = json.load(f)

with open("./../Probability_NGrams/word_1grams_probability.json", "r", encoding="utf-8") as f:
    prob_unigram_json = json.load(f)

# Convert json file to prefix→probability dictionary
prob_ngram = load_prob_Ndict(prob_bigram_json)
prob_unigram = load_prob_Unidict(prob_unigram_json)

sentences = ( 
    ["The", "economy", "has", "<gap>", "this", "year", "."],
    [
    "The", "finance", "minister", "said", "the", "<gap>", ",", "which", "had", "<gap>", "intense", "<gap>", "from","several","EU", "members", ",", "would", "be", "<gap>", "before", "the", "final", "vote", "next", "month", "."],
    [
    "After", "months", "of", "<gap>", ",", "the", "agreement", "between", "the", "two", "companies", "was", "finally", "<gap>", ",", "paving", "the", "way", "for", "joint", "<gap>", "into", "renewable", "energy", "technologies", "."
    ],
    [
    "Scientists", "reported", "that", "the", "newly", "<gap>", "exoplanet", "shows", "<gap>", "of", "an", "atmosphere", "rich", "in", "methane", ",", "a", "finding", "that", "could", "<gap>", "current", "models", "of", "planetary", "formation", "."
    ]
)

for sentence in sentences:
    filled = fill_gaps_in_sentence(sentence, n=2, nprob_dict=prob_ngram, uniprob_dict=prob_unigram)
    print(" ".join(filled))

The economy has down this year .
The finance minister said the of , which had the intense you from several EU members , would be intentional before the final vote next month .
After months of venture , the agreement between the two companies was finally lasting , paving the way for joint were into renewable energy technologies .
Scientists reported that the newly Baroness exoplanet shows Faltskog of an atmosphere rich in methane , a finding that could of current models of planetary formation .
